# Phase 2 Notebook: Candidate Generation Wikidata

## Principles
The goal is to find wikidata candidates for the mentions identified in the previous step. Example numbers:
* episodes: 2038
* publications: 2548
* seasons: 17
* persons: 10098
* topics: 10713

The goal is to be systematic and carefull, to put as little burden onto wikidata as possible. We should try to infer as much information from links as possible before we do any other searches, such as label searches or manual searches for the remaining items without candidates. 

## Approach (Simple Tree Expansion Workflow)
1. Load the broadcasting program seed(s) from `data/00_setup/broadcasting_programs.csv` and resolve their Wikidata Q-IDs.
2. For each seed Q-ID, check local cache first.
   - If cached entity data exists, use it.
   - If not cached, query Wikidata once and store the raw result as a new separate file.
3. Expand the Wikidata tree from each known Q-ID.
   - Expand outlinks (claims from the entity to other entities/properties).
   - Expand inlinks (entities that point to the current entity).
   - Persist each expansion query result as its own file before any aggregation.
4. Stage A is graph-first and authoritative: expand by graph predicates and seed/core-class rules (not by literal matches).
5. Stage B runs only for unresolved targets and performs scoped fallback string matching.
6. Eligible fallback discoveries are re-checked and re-entered into graph expansion.
7. Maintain an aggregated index for fast lookup, but always keep per-query raw files as the source of truth.
   - If an aggregate file is corrupted, rebuild it from individual query result files without data loss.

### Cache And Storage Principles
- Cache-first: never hit Wikidata if sufficient local query results already exist.
- Append-only raw query storage: one file per query result.
- Rebuildable aggregates: derived files can be regenerated from raw query files at any time.
- Idempotent reruns: rerunning the workflow should reuse existing raw files and only fetch missing queries.

## 1) Project Setup
Resolve repository paths and make the source package importable in this notebook session.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "data").exists() and (cur / "speakermining" / "src").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError("Repository root not found.")


ROOT = find_repo_root(Path.cwd())
SRC = ROOT / "speakermining" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

In [ ]:
from pathlib import Path

from IPython import get_ipython

from process.candidate_generation.wikidata.graceful_shutdown import request_termination
from process.candidate_generation.wikidata.heartbeat_monitor import emit_event_derived_heartbeat, run_with_progress_heartbeat


def install_compact_graceful_stop_handler() -> None:
    ip = get_ipython()
    if ip is None:
        return
    if globals().get("_compact_graceful_stop_installed", False):
        return

    def _compact_graceful_stop(shell, etype, evalue, tb, tb_offset=None):
        _ = (shell, tb, tb_offset)
        is_keyboard_interrupt = issubclass(etype, KeyboardInterrupt)
        is_runtime_termination = issubclass(etype, RuntimeError) and str(evalue).strip() == "Termination requested"
        if is_keyboard_interrupt or is_runtime_termination:
            request_termination()
            print("[notebook] graceful stop completed", flush=True)
            return []
        return None

    ip.set_custom_exc((KeyboardInterrupt, RuntimeError), _compact_graceful_stop)
    globals()["_compact_graceful_stop_installed"] = True
    print("[notebook] compact graceful-stop handler installed", flush=True)


install_compact_graceful_stop_handler()

## 2) Configure Workflow Parameters
Set rate limiting, cache behavior, and expansion limits before running discovery.
Use `max_queries_per_run` to control whether the notebook stays cache-only or performs live discovery queries.

In [ ]:
from process.candidate_generation.wikidata.common import normalize_query_budget, set_active_wikidata_languages
from process.candidate_generation.wikidata.backoff_learning import recommend_query_delay_from_history
from process.candidate_generation.wikidata.mention_type_config import (
    ALLOWED_MENTION_TYPES,
    assert_mention_type_snapshot_unchanged,
    normalize_mention_type_name,
    resolve_enabled_mention_types,
    snapshot_enabled_mention_types,
)

import os

# Workflow Configuration
config = {
    "max_depth": 2,                          # Max tree expansion depth (0=seeds only, 1=neighbors, 2=2nd neighbors, etc.)
    "subclass_expansion_max_depth": 3,       # Class-resolution subclass exploration depth for class_resolution_map
    "superclass_branch_discovery_max_depth": 5,  # Upward P279 branch depth from active instance classes during preflight
    "max_nodes": 500000,                     # Max total nodes to expand
    "max_queries_per_run": 5000,              # Query budget semantics: -1=unlimited, 0=no network queries (cache-first), >0=finite cap
    "max_neighbors_per_match": 200,          # Cap neighbors enqueued per matched node (0=unlimited)
    "query_timeout_seconds": 5,              # Timeout per HTTP request
    "query_delay_seconds": 0.25,             # Delay between queries (rate limiting)
    "http_max_retries": 4,                   # Retries for transient HTTP/network failures
    "http_backoff_base_seconds": 1.0,        # Base exponential backoff for retries
    "adaptive_backoff_enabled": True,        # Runtime delay adaptation based on observed backoff patterns
    "adaptive_backoff_pattern_heartbeats": 3,  # Heartbeat windows required before adaptation
    "adaptive_backoff_increase_factor": 0.05,  # +5% delay when sustained backoff pattern is detected
    "adaptive_backoff_decrease_factor": 0.01,  # -1% delay when sustained no-backoff window is observed
    "adaptive_backoff_min_delay_seconds": 0.05,
    "adaptive_backoff_max_delay_seconds": 30.0,
    "node_integrity_batch_fetch_size": 25,   # Step 6.5: number of entity IDs fetched per wbgetentities call
    "network_progress_every": 50,            # Emit runtime progress after every N network calls (0=off)
    "heartbeat_interval_seconds": 60,        # Emit a visible heartbeat line at least once per minute during long runs
    "heartbeat_window_size": 25,             # Recent events included in heartbeat summaries
    "inlinks_limit": 200,                    # Max inlinks per entity (SPARQL LIMIT)
    "cache_max_age_days": 365,               # Cache age threshold (days); older records are refreshed
    "wikidata_entity_languages": {           # Explicitly enable required languages (all False by default)
        "en": True,
        "de": True,
    },
    "fallback_enabled_mention_types": {      # Toggle fallback matching per mention type (set True/False)
        "person": False,
        "organization": False,
        "episode": False,
        "season": False,
        "topic": False,
        "broadcasting_program": False,
    },
    "fallback_prefer_class_scoped_search": False,              # Prefer class-scoped SPARQL label search before generic search
    "fallback_allow_generic_search_after_class_scoped": True,  # If class-scoped search is empty, continue with generic search
    "fallback_class_scoped_search_limit": 10,                  # Result cap for each class-scoped search call
    # Property-Value Basic Hydration (Step 2.4.2)
    "property_value_hydration_predicates": None,  # None = use default whitelist (P106, P102, P108, P21, P527, P17)
    "hydrate_all_core_instance_objects": True,     # Hydrate all objects of expanded core class instances
    "hydrate_all_objects": False,                  # WARNING: unbounded; leave False unless total unhydrated count is verified
}

# Runtime budget tracker shared across notebook steps.
config["query_budget_remaining"] = normalize_query_budget(config["max_queries_per_run"])

config["preflight_network_queries"] = 0

# Bind notebook config to materializer subclass expansion service.
os.environ["WIKIDATA_SUBCLASS_EXPANSION_MAX_DEPTH"] = str(int(config["subclass_expansion_max_depth"]))
os.environ["WIKIDATA_SUPERCLASS_BRANCH_DISCOVERY_MAX_DEPTH"] = str(int(config["superclass_branch_discovery_max_depth"]))
os.environ["WIKIDATA_ADAPTIVE_BACKOFF_ENABLED"] = "1" if bool(config.get("adaptive_backoff_enabled", True)) else "0"
os.environ["WIKIDATA_ADAPTIVE_BACKOFF_PATTERN_HEARTBEATS"] = str(int(config.get("adaptive_backoff_pattern_heartbeats", 3) or 3))
os.environ["WIKIDATA_ADAPTIVE_BACKOFF_INCREASE_FACTOR"] = str(float(config.get("adaptive_backoff_increase_factor", 0.05) or 0.05))
os.environ["WIKIDATA_ADAPTIVE_BACKOFF_DECREASE_FACTOR"] = str(float(config.get("adaptive_backoff_decrease_factor", 0.01) or 0.01))
os.environ["WIKIDATA_ADAPTIVE_BACKOFF_MIN_DELAY_SECONDS"] = str(float(config.get("adaptive_backoff_min_delay_seconds", 0.05) or 0.05))
os.environ["WIKIDATA_ADAPTIVE_BACKOFF_MAX_DELAY_SECONDS"] = str(float(config.get("adaptive_backoff_max_delay_seconds", 30.0) or 30.0))

backoff_delay_guidance = recommend_query_delay_from_history(
    ROOT,
    configured_delay_seconds=float(config.get("query_delay_seconds", 0.0) or 0.0),
)
config["backoff_delay_guidance"] = backoff_delay_guidance

if bool(backoff_delay_guidance.get("known_backoff_prone", False)):
    print(
        "[notebook][warning] query_delay_seconds has historical backoff evidence: "
        f"{backoff_delay_guidance.get('message', '')}"
    )
elif int(backoff_delay_guidance.get("samples", 0) or 0) > 0:
    print(
        "[notebook] historical delay evidence found: "
        f"samples={int(backoff_delay_guidance.get('samples', 0) or 0)} "
        f"backoffs={int(backoff_delay_guidance.get('backoffs', 0) or 0)} "
        f"rate={float(backoff_delay_guidance.get('backoff_rate', 0.0) or 0.0):.3f}"
    )

# Raises ValueError('Please define at least one language') when all flags are False.
enabled_wikidata_languages = set_active_wikidata_languages(config["wikidata_entity_languages"])

config["wikidata_enabled_languages"] = list(enabled_wikidata_languages)

# Resolve fallback mention types once; downstream cells must reuse this exact value.
config["fallback_enabled_mention_types_resolved"] = resolve_enabled_mention_types(
    config.get("fallback_enabled_mention_types", {})
)
config["_fallback_enabled_mention_types_snapshot"] = snapshot_enabled_mention_types(
    config.get("fallback_enabled_mention_types", {})
)

print("Workflow Configuration:")
for key, value in config.items():
    if key.startswith("_"):
        continue
    if key == "fallback_enabled_mention_types" and isinstance(value, dict):
        print(f"  {key}: {value}")
        print(f"  fallback_enabled (resolved): {config['fallback_enabled_mention_types_resolved']}")
    elif key == "wikidata_entity_languages" and isinstance(value, dict):
        print(f"  {key}: {value}")
        print(f"  wikidata_enabled_languages (resolved): {config['wikidata_enabled_languages']}")
    elif key == "backoff_delay_guidance" and isinstance(value, dict):
        print(
            "  backoff_delay_guidance: "
            f"samples={int(value.get('samples', 0) or 0)} "
            f"backoffs={int(value.get('backoffs', 0) or 0)} "
            f"known_backoff_prone={bool(value.get('known_backoff_prone', False))} "
            f"recommended_delay_seconds={value.get('recommended_delay_seconds', config.get('query_delay_seconds'))}"
        )
    else:
        print(f"  {key}: {value}")

print(f"  WIKIDATA_SUBCLASS_EXPANSION_MAX_DEPTH (env): {os.environ['WIKIDATA_SUBCLASS_EXPANSION_MAX_DEPTH']}")
print(f"  WIKIDATA_SUPERCLASS_BRANCH_DISCOVERY_MAX_DEPTH (env): {os.environ['WIKIDATA_SUPERCLASS_BRANCH_DISCOVERY_MAX_DEPTH']}")
print(f"  WIKIDATA_ADAPTIVE_BACKOFF_ENABLED (env): {os.environ['WIKIDATA_ADAPTIVE_BACKOFF_ENABLED']}")
print(f"  WIKIDATA_ADAPTIVE_BACKOFF_PATTERN_HEARTBEATS (env): {os.environ['WIKIDATA_ADAPTIVE_BACKOFF_PATTERN_HEARTBEATS']}")

## 2.4) Preflight Subclass Expansion (Run First, Budget-Safe)
Refresh class hierarchy and subclass-resolution artifacts before graph expansion starts.

This preflight is local-materialization only and does not consume Stage A query budget.
Run this before the class-hierarchy clarification and before any Stage A execution.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import importlib
import pandas as pd

from process.candidate_generation.wikidata.bootstrap import load_core_classes
from process.candidate_generation.wikidata.common import canonical_qid, normalize_query_budget
import process.candidate_generation.wikidata.schemas as schemas_module
import process.candidate_generation.wikidata.event_log as event_log_module
import process.candidate_generation.wikidata.cache as cache_module
import process.candidate_generation.wikidata.inlinks as inlinks_module
import process.candidate_generation.wikidata.materializer as materializer_module

schemas_module = importlib.reload(schemas_module)
event_log_module = importlib.reload(event_log_module)
cache_module = importlib.reload(cache_module)
inlinks_module = importlib.reload(inlinks_module)
materializer_module = importlib.reload(materializer_module)
crawl_subclass_expansion = materializer_module.crawl_subclass_expansion

print("[notebook] Step 2.4 start: explicit subclass expansion crawl")
print(
    "[notebook] Step 2.4 policy: breadth-first P279 crawl from each core class, cache-first, "
    "depth-bounded, budget-bounded"
)

budget_before = normalize_query_budget(config.get("query_budget_remaining", config.get("max_queries_per_run", 0)))
print(f"  budget_before_preflight: {budget_before}")

preflight_run_id = f"preflight-subclass-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
preflight_stats = crawl_subclass_expansion(
    ROOT,
    run_id=preflight_run_id,
    max_depth=int(config.get("subclass_expansion_max_depth", 2) or 2),
    superclass_branch_discovery_max_depth=int(config.get("superclass_branch_discovery_max_depth", 5) or 0),
    query_budget_remaining=budget_before,
    cache_max_age_days=int(config.get("cache_max_age_days", 365) or 365),
    query_timeout_seconds=int(config.get("query_timeout_seconds", 5) or 5),
    query_delay_seconds=float(config.get("query_delay_seconds", 0.0) or 0.0),
    progress_every_calls=int(config.get("network_progress_every", 50) or 0),
    progress_every_seconds=float(config.get("heartbeat_interval_seconds", 60) or 60),
    http_max_retries=int(config.get("http_max_retries", 4) or 0),
    http_backoff_base_seconds=float(config.get("http_backoff_base_seconds", 1.0) or 0.0),
    page_limit=int(config.get("inlinks_limit", 200) or 200),
)
preflight_network_queries = int(preflight_stats.get("network_queries", 0))

if budget_before >= 0:
    budget_after = max(0, budget_before - preflight_network_queries)
else:
    budget_after = -1

config["preflight_network_queries"] = preflight_network_queries
config["query_budget_remaining"] = budget_after

print("[notebook] Step 2.4 complete")
print(f"  preflight_run_id: {preflight_run_id}")
print(f"  subclass_expansion_max_depth: {preflight_stats.get('subclass_expansion_max_depth')}")
print(f"  class_resolution_rows: {preflight_stats.get('class_resolution_rows')}")
print(f"  class_resolution_conflict_rows: {preflight_stats.get('class_resolution_conflict_rows')}")
print(f"  superclass_branch_discovery_max_depth: {preflight_stats.get('superclass_branch_discovery_max_depth')}")
print(f"  superclass_branch_connected_active_classes: {preflight_stats.get('superclass_branch_connected_active_classes')}")
print(f"  superclass_branch_nodes: {preflight_stats.get('superclass_branch_nodes')}")
print(f"  superclass_branch_network_calls: {preflight_stats.get('superclass_branch_network_calls')}")
print(f"  preflight_network_queries: {preflight_network_queries}")
print(f"  budget_after_preflight: {budget_after}")

class_resolution_map_path = ROOT / "data" / "20_candidate_generation" / "wikidata" / "projections" / "class_resolution_map.csv"
core_rows = load_core_classes(ROOT)
core_lookup = {
    canonical_qid(str(row.get("wikidata_id", "") or "")): str(row.get("filename", "") or "")
    for row in core_rows
    if canonical_qid(str(row.get("wikidata_id", "") or ""))
}

if class_resolution_map_path.exists():
    class_resolution_map_df = pd.read_csv(class_resolution_map_path)
    per_core_rows = []
    for core_qid, core_filename in core_lookup.items():
        subset = class_resolution_map_df[class_resolution_map_df.get("resolved_core_class_id", "") == core_qid].copy()
        subclass_count = int(subset["class_id"].nunique()) if not subset.empty and "class_id" in subset.columns else 0
        max_depth_observed = int(subset["resolution_depth"].dropna().astype(float).max()) if not subset.empty and "resolution_depth" in subset.columns and subset["resolution_depth"].notna().any() else -1
        conflict_rows = int(subset["conflict_flag"].fillna(False).astype(bool).sum()) if not subset.empty and "conflict_flag" in subset.columns else 0
        per_core_rows.append(
            {
                "core_class": core_filename,
                "core_qid": core_qid,
                "subclass_count": subclass_count,
                "max_depth_observed": max_depth_observed,
                "conflict_rows": conflict_rows,
            }
        )

    subclass_stats_df = pd.DataFrame(per_core_rows).sort_values(["subclass_count", "core_class"], ascending=[False, True]).reset_index(drop=True)
    print("\nSubclass expansion detail by core class:")
    display(subclass_stats_df)
else:
    print(f"class_resolution_map.csv not found at {class_resolution_map_path}")

print("  note: Step 6 will use config['query_budget_remaining'] as its budget input")

#### 2.4.1 Explore conflict rows

In [ ]:
import importlib

import process.candidate_generation.wikidata.conflict_analysis as conflict_analysis_module

conflict_analysis_module = importlib.reload(conflict_analysis_module)
inspect_class_resolution_conflicts = conflict_analysis_module.inspect_class_resolution_conflicts
print_conflict_report = conflict_analysis_module.print_conflict_report

print("[notebook] Step 2.4.1 start: discover conflict patterns from all rows")

label_language_preference = "en"
for lang_key, enabled in (config.get("wikidata_entity_languages", {}) or {}).items():
    if bool(enabled):
        label_language_preference = str(lang_key).strip().lower() or "en"
        break

print(f"[notebook] Step 2.4.1 label language: {label_language_preference}")

conflict_report = inspect_class_resolution_conflicts(
    ROOT,
    min_cluster_size=2,
    top_clusters=12,
    label_language_preference=label_language_preference,
)
print_conflict_report(conflict_report, repo_root=ROOT)

print("\n[notebook] Step 2.4.1 complete")

#### 2.4.2) Property-Value Basic Hydration

Ensures every QID referenced as a property value on an expanded core-class instance has label/description/alias/P31/P279 data in the entity store.

Runs after subclass preflight (Step 2.4). Hydrates in two scopes:
1. Objects of whitelisted predicates (`property_value_hydration_predicates` in config — default: P106, P102, P108, P21, P527, P17).
2. All objects of expanded non-class (instance) nodes when `hydrate_all_core_instance_objects=True`.

QIDs that are `inactive_guarded` (in store but empty) are activated. QIDs absent from the store are fetched and upserted. Already-hydrated QIDs are skipped. All network calls are cache-first, rate-limited, and budget-bounded.

In [ ]:
import importlib
from datetime import datetime, timezone

import process.candidate_generation.wikidata.materializer as materializer_module

materializer_module = importlib.reload(materializer_module)
run_property_value_hydration = materializer_module.run_property_value_hydration

print("[notebook] Step 2.4.2 start: property-value basic hydration")

prop_hydration_run_id = f"prop-hydration-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

prop_hydration_stats = run_property_value_hydration(
    ROOT,
    run_id=prop_hydration_run_id,
    query_budget_remaining=config.get("query_budget_remaining", -1),
    cache_max_age_days=int(config.get("cache_max_age_days", 365) or 365),
    query_timeout_seconds=int(config.get("query_timeout_seconds", 5) or 5),
    query_delay_seconds=float(config.get("query_delay_seconds", 0.0) or 0.0),
    progress_every_calls=int(config.get("network_progress_every", 50) or 0),
    http_max_retries=int(config.get("http_max_retries", 4) or 0),
    http_backoff_base_seconds=float(config.get("http_backoff_base_seconds", 1.0) or 0.0),
    hydration_predicates=config.get("property_value_hydration_predicates"),
    hydrate_all_core_instance_objects=bool(config.get("hydrate_all_core_instance_objects", True)),
    hydrate_all_objects=bool(config.get("hydrate_all_objects", False)),
)

prop_hydration_queries_used = int(prop_hydration_stats.get("network_queries", 0))
if int(config.get("query_budget_remaining", -1)) >= 0:
    config["query_budget_remaining"] = max(0, config["query_budget_remaining"] - prop_hydration_queries_used)

print("[notebook] Step 2.4.2 complete")
print(f"  run_id: {prop_hydration_run_id}")
print(f"  status: {prop_hydration_stats.get('status')}")
print(f"  total_candidates: {prop_hydration_stats.get('total_candidates')}")
print(f"  inactive_guarded_activated: {prop_hydration_stats.get('inactive_guarded_activated')}")
print(f"  absent_hydrated: {prop_hydration_stats.get('absent_hydrated')}")
print(f"  total_hydrated: {prop_hydration_stats.get('total_hydrated')}")
print(f"  network_queries: {prop_hydration_queries_used}")
print(f"  budget_remaining: {config['query_budget_remaining']}")

#### 2.4.3) Second-Pass Superclass Branch Discovery

Re-runs the subclass preflight with an extended active-class seed set that includes property-value hydration predicate objects (P106 occupation, P102 party, P108 employer, P17 country, P527 has parts) in addition to P31 (instance-of).

**Why this is needed:** Step 2.4 seeds upward branch discovery only from P31 objects. Occupation QIDs (e.g. biologist Q864503) appear in P106 triples on persons — invisible to the P31 seed, never walked upward to the role core class. Step 2.4.2 hydrated these nodes; this step connects them to the class_resolution_map so the full chain (biologist → scientist → … → occupation → position → role) is registered.

Pass 1 (top-down BFS at the same depth) is cache-first and fast. Pass 2 (upward branch walk) seeds from the extended predicate set and connects newly hydrated occupation nodes to role via `combined_paths`.

In [ ]:
import importlib
from datetime import datetime, timezone

import process.candidate_generation.wikidata.materializer as materializer_module

materializer_module = importlib.reload(materializer_module)
crawl_subclass_expansion = materializer_module.crawl_subclass_expansion

print("[notebook] Step 2.4.3 start: second-pass superclass branch discovery")

second_pass_run_id = f"preflight-subclass-pass2-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

_hydration_predicates = config.get("property_value_hydration_predicates") or ["P106", "P102", "P108", "P21", "P527", "P17"]

second_pass_stats = crawl_subclass_expansion(
    ROOT,
    run_id=second_pass_run_id,
    max_depth=int(config.get("subclass_expansion_max_depth", 2) or 2),
    superclass_branch_discovery_max_depth=int(config.get("superclass_branch_discovery_max_depth", 5) or 0),
    query_budget_remaining=config.get("query_budget_remaining", -1),
    cache_max_age_days=int(config.get("cache_max_age_days", 365) or 365),
    query_timeout_seconds=int(config.get("query_timeout_seconds", 5) or 5),
    query_delay_seconds=float(config.get("query_delay_seconds", 0.0) or 0.0),
    progress_every_calls=int(config.get("network_progress_every", 50) or 0),
    progress_every_seconds=float(config.get("heartbeat_interval_seconds", 60) or 60),
    http_max_retries=int(config.get("http_max_retries", 4) or 0),
    http_backoff_base_seconds=float(config.get("http_backoff_base_seconds", 1.0) or 0.0),
    page_limit=int(config.get("inlinks_limit", 200) or 200),
    additional_active_class_predicates=tuple(p for p in _hydration_predicates if p),
)

second_pass_queries = int(second_pass_stats.get("network_queries", 0))
if int(config.get("query_budget_remaining", -1)) >= 0:
    config["query_budget_remaining"] = max(0, config["query_budget_remaining"] - second_pass_queries)

print("[notebook] Step 2.4.3 complete")
print(f"  run_id: {second_pass_run_id}")
print(f"  class_resolution_rows: {second_pass_stats.get('class_resolution_rows')}")
print(f"  superclass_branch_connected_active_classes: {second_pass_stats.get('superclass_branch_connected_active_classes')}")
print(f"  superclass_branch_nodes: {second_pass_stats.get('superclass_branch_nodes')}")
print(f"  active_classes_from_triples: {second_pass_stats.get('active_classes_from_triples')}")
print(f"  network_queries: {second_pass_queries}")
print(f"  budget_remaining: {config['query_budget_remaining']}")

## 2.5) Class Hierarchy Clarification: Core vs Root Classes

**Critical distinction for query efficiency and graph scope:**

- **Core classes** (what we care about): Person (Q5), Organization (Q43229), Episode (Q25379), Season (Q3464665), Topic (labels vary), Broadcasting Program (Q1364556)
  - These define our primary entity types of interest
  - We expand their neighborhoods and instances systematically
  - Instances of these classes are our primary discovery targets

- **Root classes** (what to avoid over-expanding): Entity (Q35120), Thing (Q1), owl:Thing
  - Nearly everything in Wikidata is eventually an instance or subclass of these universal classes
  - Recursively expanding from root classes can enumerate thousands of low-value nodes
  - We use root classes ONLY for contrast and avoidance logic, not for discovery boundaries

**Why this matters:**
Conflating a root class with a core class means exploring thousands of nodes and their neighbors despite having no direct interest in them. This can multiply network queries and discovery time without yielding meaningful candidates.

**Runtime validation:**
This cell validates that core_class_qids and root_class_qids are disjoint and correctly understood before graph expansion begins.

In [ ]:
from process.candidate_generation.wikidata.bootstrap import load_core_classes
from process.candidate_generation.wikidata.common import canonical_qid

print("[notebook] Step 2.5 start: class hierarchy clarification")

# Load configured core classes from setup
core_classes = load_core_classes(ROOT)
core_class_qids = {canonical_qid(row.get('wikidata_id', '')) for row in core_classes if canonical_qid(row.get('wikidata_id', ''))}

# Define root classes (universal superclasses that should NOT be expansion boundaries)
# These exist to contrast with core classes and should be referenced only for avoidance logic
root_class_qids = {'Q35120', 'Q1'}  # Entity, Thing

print("\n--- Class Hierarchy Definitions ---")
print(f"\nCore classes (PRIMARY DISCOVERY TARGETS): {len(core_class_qids)}")
for qid in sorted(core_class_qids):
    row = next((r for r in core_classes if canonical_qid(r.get('wikidata_id', '')) == qid), {})
    label = row.get('filename', qid)
    print(f"  - {qid}: {label}")

print(f"\nRoot classes (UNIVERSAL SUPERCLASSES - AVOID OVER-EXPANSION): {len(root_class_qids)}")
print(f"  - Q35120: Entity")
print(f"  - Q1: Thing")

# Validate that core and root classes are disjoint (no overlap)
overlap = core_class_qids & root_class_qids
if overlap:
    raise ValueError(
        f"FATAL: Core classes and root classes MUST be disjoint (no overlap).\n"
        f"Found overlap: {overlap}\n"
        f"This is a configuration error that would cause incorrect graph boundary behavior."
    )

print(f"\n[OK] Validation passed: core_class_qids and root_class_qids are disjoint.")
print(f"\nScope contract:")
print(f"  - Instances of core classes are discovery targets -> will trigger expansion")
print(f"  - Instances of root classes are universal -> expansion limited by core-class-connection rules")
print(f"  - Non-core subclasses are discoverable but not primary targets")

# Store for downstream use
config["core_class_qids"] = core_class_qids
config["root_class_qids"] = root_class_qids

print(f"\n[notebook] Step 2.5 complete")

## 3) Decide Resume Mode
Define whether this run should append from latest checkpoint or revert one seed and continue.

Safety policy: use non-destructive resume modes only to preserve fetched query history.

Mode implications (must-read before running):
- `append`: continue from the latest checkpoint lineage without deleting fetched-query history; deterministic seed scan starts at program 1 and skips completed seeds.
- `revert`: restore the previous checkpoint snapshot (one checkpoint back) and then continue in append semantics from that restored state.

Unwanted consequences to avoid:
- Use `append` for normal continuation and incremental progress.
- Use `revert` only when the latest checkpoint is suspected to be invalid and you intentionally accept rollback of post-checkpoint progress.

In [ ]:
from process.candidate_generation.wikidata.checkpoint import decide_resume_mode

# Mode selection policy (clarity over convenience):
# - append: keep fetched-query history, reuse latest checkpoint lineage, and continue deterministic seed scan
#           from program 1 while skipping already completed seeds.
# - revert: restore previous checkpoint snapshot (one checkpoint back), then continue with append semantics.
# Use revert only when latest checkpoint state is suspected invalid and rollback is intentional.
resume_mode = "append"  # options: append, revert

resume_decision = decide_resume_mode(ROOT, resume_mode)
print("Resume decision:")
print(resume_decision)

## 4) Bootstrap and Load Broadcasting Program Seeds
Initialize required artifacts from setup data and load seed instances for expansion.

In [ ]:
from process.candidate_generation.broadcasting_program import load_broadcasting_program_seeds
from process.candidate_generation.wikidata.bootstrap import initialize_bootstrap_files, load_core_classes, load_seed_instances
from process.candidate_generation.wikidata.common import canonical_qid

setup_core_classes = load_core_classes(ROOT)
setup_seeds, skipped_seed_rows = load_seed_instances(ROOT)
initialize_bootstrap_files(ROOT, setup_core_classes, setup_seeds)

seeds = setup_seeds or load_broadcasting_program_seeds(ROOT)
seed_qids = [canonical_qid(seed.get("wikidata_id", "")) for seed in seeds]
seed_qids = [qid for qid in seed_qids if qid]

print(f"Bootstrap initialized with {len(setup_core_classes)} core classes and {len(setup_seeds)} valid seed rows")
print(f"Skipped seed rows due to invalid wikidata_id: {len(skipped_seed_rows)}")
print(f"Loaded {len(seeds)} broadcasting programs as seeds")
for seed in seeds:
    qid = canonical_qid(seed.get("wikidata_id", ""))
    if qid:
        print(f"  - {seed.get('label', seed.get('name', 'N/A'))} ({qid})")


## 5) Build Mention Targets From Phase 2 Lookup Outputs
Target construction is delegated to a process module for maintainability.
Sources:
- `data/20_candidate_generation/episodes.csv` for episode, season, person, topic, and organization-like publication program mentions.
- `data/20_candidate_generation/broadcasting_programs.csv` for explicit broadcasting program entities (show-level).

In [ ]:
from process.candidate_generation.wikidata.candidate_targets import build_targets_from_phase2_lookup

phase2_dir = ROOT / "data" / "20_candidate_generation"
required_inputs = [
    phase2_dir / "episodes.csv",
    phase2_dir / "broadcasting_programs.csv",
]
missing_inputs = [path for path in required_inputs if not path.exists()]

if missing_inputs:
    missing_list = "\n".join(f"- {path}" for path in missing_inputs)
    raise FileNotFoundError(
        "Notebook 21 prerequisite missing: required Phase 2 candidate-generation inputs were not found.\n"
        f"Missing files:\n{missing_list}\n\n"
        "Please run speakermining/src/process/notebooks/20_candidate_generation_wikibase.ipynb first "
        "to generate these files, then rerun this cell."
    )

all_target_rows, mention_stats, target_df = build_targets_from_phase2_lookup(ROOT)
total_mentions = len(all_target_rows)

print("Loaded matching targets from Phase 2 lookup outputs")
print(f"target rows for matching: {total_mentions}")
print("\nMention Targets by Type:")
print(mention_stats.to_string())

print("\nTargets:")
display(target_df)

## 6) Execute Graph-First Expansion Stage
Run deterministic seed-ordered graph expansion (authoritative stage).
In append mode, the stage always scans from program 1, skips already completed seeds, and resumes work at the first unfinished seed.
Checkpoint manifests are written per seed boundary, while expensive projection materialization runs once at final stage completion.

In [ ]:
from time import perf_counter

from process.candidate_generation.wikidata.expansion_engine import ExpansionConfig, run_graph_expansion_stage
from process.candidate_generation.wikidata.bootstrap import load_core_classes
from process.candidate_generation.wikidata.common import normalize_query_budget
from process.candidate_generation.wikidata.notebook_orchestrator import resolve_heartbeat_settings

if "preflight_stats" not in globals():
    raise RuntimeError(
        "Step 2.4 must run before Step 6. Run the preflight subclass expansion cell first "
        "so class hierarchy and subclass resolution artifacts are refreshed before Stage A budget is spent."
    )

configured_subclass_depth = int(config.get("subclass_expansion_max_depth", 2) or 2)
preflight_depth = int(preflight_stats.get("subclass_expansion_max_depth", -1) or -1)
if preflight_depth != configured_subclass_depth:
    raise RuntimeError(
        f"Step 2.4 preflight depth mismatch: configured={configured_subclass_depth}, observed={preflight_depth}. "
        "Re-run Step 2.4 before Step 6."
    )

remaining_budget_for_stage_a = normalize_query_budget(
    config.get("query_budget_remaining", config.get("max_queries_per_run", 0))
)

print("[notebook] Step 6 start: graph-first expansion")
print(f"Target rows from episodes.csv: {len(all_target_rows)}")
print(
    "[notebook] Step 6 prerequisite satisfied: preflight subclass expansion already executed "
    f"at depth={preflight_depth}"
 )
print(f"[notebook] Step 6 budget input (remaining): {remaining_budget_for_stage_a}")

existing_heartbeat_settings = globals().get("heartbeat_settings")
heartbeat_settings = resolve_heartbeat_settings(config, existing_heartbeat_settings if isinstance(existing_heartbeat_settings, dict) else None)

expansion_config = ExpansionConfig(
    max_depth=config["max_depth"],
    max_nodes=config["max_nodes"],
    total_query_budget=remaining_budget_for_stage_a,
    per_seed_query_budget=remaining_budget_for_stage_a,
    query_timeout_seconds=config["query_timeout_seconds"],
    query_delay_seconds=config["query_delay_seconds"],
    network_progress_every=int(config.get("network_progress_every", 50) or 0),
    inlinks_limit=config["inlinks_limit"],
    cache_max_age_days=config["cache_max_age_days"],
    max_neighbors_per_node=config["max_neighbors_per_match"],
)

step6_t0 = perf_counter()
print("[notebook] -> run_graph_expansion_stage")
result = run_with_progress_heartbeat(
    ROOT,
    phase="stage_a_graph_expansion",
    work_label="run_graph_expansion_stage",
    heartbeat_interval_seconds=int(heartbeat_settings["interval_seconds"]),
    heartbeat_window_size=int(heartbeat_settings["window_size"]),
    work_fn=lambda: run_graph_expansion_stage(
        ROOT,
        seeds=seeds,
        targets=all_target_rows,
        core_class_qids={canonical_qid(row.get('wikidata_id', '')) for row in load_core_classes(ROOT) if canonical_qid(row.get('wikidata_id', ''))},
        config=expansion_config,
        requested_mode=resume_decision["mode"],
    ),
)
step6_elapsed = perf_counter() - step6_t0
summary = result.checkpoint_stats

stage_a_queries_used = int(summary.get("stage_a_network_queries_this_run", 0) or 0)
if remaining_budget_for_stage_a >= 0:
    config["query_budget_remaining"] = max(0, remaining_budget_for_stage_a - stage_a_queries_used)
else:
    config["query_budget_remaining"] = -1

print("\n[notebook] Step 6 complete")
print(f"[notebook] Step 6 elapsed seconds: {step6_elapsed:.2f}")
print(f"[notebook] Step 6 queries used: {stage_a_queries_used}")
print(f"[notebook] Budget remaining after Step 6: {config['query_budget_remaining']}")
print("Execution Summary:")
print("=" * 50)
for key, value in summary.items():
    if not key.endswith("_csv"):
        print(f"  {key:<30} {value}")

In [ ]:
from process.candidate_generation.wikidata.notebook_orchestrator import resolve_heartbeat_settings

existing_heartbeat_settings = globals().get("heartbeat_settings")
heartbeat_settings = resolve_heartbeat_settings(config, existing_heartbeat_settings if isinstance(existing_heartbeat_settings, dict) else None)

stage6_heartbeat = emit_event_derived_heartbeat(
    ROOT,
    phase="step_6_graph_expansion",
    window_size=int(heartbeat_settings["window_size"]),
)
print(f"[notebook] graph expansion heartbeat summary: {stage6_heartbeat}")

## 6.5) Run Node Integrity Pass
Verify and repair graph integrity before fallback:
- ensure known nodes have minimal discovery payload
- expand discovered nodes that are eligible but still unexpanded
- persist diagnostics on regenerated nodes for follow-up bug analysis

In [ ]:
import json
from datetime import datetime, timezone
import pandas as pd

from process.candidate_generation.wikidata.node_integrity import (
    NodeIntegrityConfig,
    run_node_integrity_pass,
)
from process.candidate_generation.wikidata.notebook_orchestrator import (
    build_node_integrity_budget_plan,
    resolve_heartbeat_settings,
    resolve_stage_a_queries_this_run,
)

print("[notebook] Step 6.5 start: node integrity pass")
raw_max_queries_per_run = int(config.get("query_budget_remaining", config.get("max_queries_per_run", 0)) or 0)
existing_heartbeat_settings = globals().get("heartbeat_settings")
heartbeat_settings = resolve_heartbeat_settings(config, existing_heartbeat_settings if isinstance(existing_heartbeat_settings, dict) else None)

stage_a_queries_this_run = resolve_stage_a_queries_this_run(result.checkpoint_stats)

node_integrity_budget_plan = build_node_integrity_budget_plan(
    raw_max_queries_per_run,
    stage_a_queries_this_run,
 )
node_integrity_discovery_budget = int(node_integrity_budget_plan['discovery_query_budget'])
node_integrity_total_expansion_budget = int(node_integrity_budget_plan['total_expansion_query_budget'])
node_integrity_per_node_budget = int(node_integrity_budget_plan['per_node_expansion_query_budget'])
node_integrity_budget_label = str(node_integrity_budget_plan['label'])

node_integrity_http_max_retries = int(config.get("http_max_retries", 4) or 0)
node_integrity_http_backoff_base_seconds = float(config.get("http_backoff_base_seconds", 1.0) or 0.0)

print(
    f"[notebook] Step 6.5 budget planning: stage_a_queries_this_run={stage_a_queries_this_run}, "
    f"node_integrity_budget={node_integrity_budget_label}"
)
print(
    f"[notebook] Step 6.5 network retry policy: "
    f"http_max_retries={node_integrity_http_max_retries}, "
    f"http_backoff_base_seconds={node_integrity_http_backoff_base_seconds}"
)
node_integrity_config = NodeIntegrityConfig(
    cache_max_age_days=config["cache_max_age_days"],
    query_timeout_seconds=config["query_timeout_seconds"],
    query_delay_seconds=config["query_delay_seconds"],
    http_max_retries=node_integrity_http_max_retries,
    http_backoff_base_seconds=node_integrity_http_backoff_base_seconds,
    network_progress_every=int(config.get("network_progress_every", 50) or 0),
    discovery_query_budget=node_integrity_discovery_budget,
    per_node_expansion_query_budget=node_integrity_per_node_budget,
    total_expansion_query_budget=node_integrity_total_expansion_budget,
    inlinks_limit=config["inlinks_limit"],
    discovery_batch_fetch_size=int(config.get("node_integrity_batch_fetch_size", 1) or 1),
    max_nodes_to_expand=0,
)

node_integrity_t0 = perf_counter()
node_integrity_result = run_with_progress_heartbeat(
    ROOT,
    phase="step_6_5_node_integrity",
    work_label="run_node_integrity_pass",
    heartbeat_interval_seconds=int(heartbeat_settings["interval_seconds"]),
    heartbeat_window_size=int(heartbeat_settings["window_size"]),
    work_fn=lambda: run_node_integrity_pass(
        ROOT,
        config=node_integrity_config,
        seed_qids={canonical_qid(seed.get("wikidata_id", "")) for seed in seeds if canonical_qid(seed.get("wikidata_id", ""))},
        core_class_qids={canonical_qid(row.get("wikidata_id", "")) for row in load_core_classes(ROOT) if canonical_qid(row.get("wikidata_id", ""))},
    ),
)
node_integrity_elapsed = perf_counter() - node_integrity_t0

run_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
diagnostics_dir = ROOT / "data" / "20_candidate_generation" / "wikidata" / "node_integrity"
docs_dir = ROOT / "documentation" / "context" / "node_integrity"
diagnostics_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)

summary_record = {
    "run_timestamp_utc": run_ts,
    "known_qids": node_integrity_result.known_qids,
    "checked_qids": node_integrity_result.checked_qids,
    "repaired_discovery_qids": node_integrity_result.repaired_discovery_qids,
    "newly_discovered_qids": len(node_integrity_result.newly_discovered_qids),
    "eligible_unexpanded_qids": len(node_integrity_result.eligible_unexpanded_qids),
    "expanded_qids": len(node_integrity_result.expanded_qids),
    "network_queries_discovery": node_integrity_result.network_queries_discovery,
    "network_queries_expansion": node_integrity_result.network_queries_expansion,
    "total_network_queries": node_integrity_result.total_network_queries,
    "timeout_warnings": node_integrity_result.timeout_warnings,
    "stop_reason": node_integrity_result.stop_reason,
    "elapsed_seconds": round(node_integrity_elapsed, 3),
}

event_rows = []
for qid in sorted(node_integrity_result.repaired_qids):
    event_rows.append({"qid": qid, "event": "repaired_discovery"})
for qid in sorted(node_integrity_result.newly_discovered_qids):
    event_rows.append({"qid": qid, "event": "newly_discovered"})
for qid in sorted(node_integrity_result.expanded_qids):
    event_rows.append({"qid": qid, "event": "expanded_by_integrity"})

events_df = pd.DataFrame(event_rows, columns=["qid", "event"])
if not events_df.empty:
    events_df = events_df.sort_values(["event", "qid"]).reset_index(drop=True)

summary_json_path = diagnostics_dir / f"node_integrity_summary_{run_ts}.json"
summary_csv_path = diagnostics_dir / f"node_integrity_summary_{run_ts}.csv"
events_csv_path = diagnostics_dir / f"node_integrity_events_{run_ts}.csv"
transitions_jsonl_path = diagnostics_dir / f"node_integrity_transitions_{run_ts}.jsonl"
doc_md_path = docs_dir / f"node_integrity_{run_ts}.md"

summary_json_path.write_text(
    json.dumps({"summary": summary_record, "materialize_stats": node_integrity_result.materialize_stats}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
pd.DataFrame([summary_record]).to_csv(summary_csv_path, index=False, encoding="utf-8")
events_df.to_csv(events_csv_path, index=False, encoding="utf-8")

# Write eligibility transitions to JSONL
if node_integrity_result.eligibility_transitions:
    with open(transitions_jsonl_path, "w", encoding="utf-8") as f:
        for transition_row in node_integrity_result.eligibility_transitions:
            f.write(json.dumps(transition_row, ensure_ascii=False) + "\n")

md_lines = [
    "# Node Integrity Report",
    "",
    f"timestamp_utc: {run_ts}",
    "",
    "## Summary",
    "",
    f"- known_qids: {summary_record['known_qids']}",
    f"- checked_qids: {summary_record['checked_qids']}",
    f"- repaired_discovery_qids: {summary_record['repaired_discovery_qids']}",
    f"- newly_discovered_qids: {summary_record['newly_discovered_qids']}",
    f"- eligible_unexpanded_qids: {summary_record['eligible_unexpanded_qids']}",
    f"- expanded_qids: {summary_record['expanded_qids']}",
    f"- network_queries_discovery: {summary_record['network_queries_discovery']}",
    f"- network_queries_expansion: {summary_record['network_queries_expansion']}",
    f"- total_network_queries: {summary_record['total_network_queries']}",
    f"- timeout_warnings: {summary_record['timeout_warnings']}",
    f"- stop_reason: {summary_record['stop_reason']}",
    f"- elapsed_seconds: {summary_record['elapsed_seconds']}",
    "",
    "## Artifact Paths",
    "",
    f"- summary_json: {summary_json_path.relative_to(ROOT)}",
    f"- summary_csv: {summary_csv_path.relative_to(ROOT)}",
    f"- events_csv: {events_csv_path.relative_to(ROOT)}",
    f"- transitions_jsonl: {transitions_jsonl_path.relative_to(ROOT)}",
    "",
    "## Regenerated / Expanded QIDs",
    "",
]
if events_df.empty:
    md_lines.append("- none")
else:
    for _, row in events_df.iterrows():
        md_lines.append(f"- {row['event']}: {row['qid']}")

doc_md_path.write_text("\n".join(md_lines) + "\n", encoding="utf-8")

print(f"[notebook] Step 6.5 complete in {node_integrity_elapsed:.2f}s")
print("Node integrity summary:")
for key in [
    "known_qids",
    "checked_qids",
    "repaired_discovery_qids",
    "newly_discovered_qids",
    "eligible_unexpanded_qids",
    "expanded_qids",
    "network_queries_discovery",
    "network_queries_expansion",
    "total_network_queries",
    "timeout_warnings",
    "stop_reason",
]:
    print(f"  {key}: {summary_record[key]}")

print("\nPersistent diagnostics written:")
print(f"  - {summary_json_path.relative_to(ROOT)}")
print(f"  - {summary_csv_path.relative_to(ROOT)}")
print(f"  - {events_csv_path.relative_to(ROOT)}")
if node_integrity_result.eligibility_transitions:
    print(f"  - {transitions_jsonl_path.relative_to(ROOT)}")
print(f"  - {doc_md_path.relative_to(ROOT)}")

if events_df.empty:
    print("\nNo regenerated or expanded nodes were required in this pass.")
else:
    print("\nNodes touched by integrity pass:")
    display(events_df)

if node_integrity_result.eligibility_transitions:
    print(f"\nEligibility transitions detected: {len(node_integrity_result.eligibility_transitions)}")

In [ ]:
from process.candidate_generation.wikidata.notebook_orchestrator import resolve_heartbeat_settings

existing_heartbeat_settings = globals().get("heartbeat_settings")
heartbeat_settings = resolve_heartbeat_settings(config, existing_heartbeat_settings if isinstance(existing_heartbeat_settings, dict) else None)

node_integrity_heartbeat = emit_event_derived_heartbeat(
    ROOT,
    phase="step_6_5_node_integrity",
    window_size=int(heartbeat_settings["window_size"]),
)
print(f"[notebook] node integrity heartbeat summary: {node_integrity_heartbeat}")


## 7) Build Unresolved Handoff
Prepare unresolved targets and class-scope hints for fallback stage.

In [ ]:
unresolved_targets = result.unresolved_targets
core_class_rows = load_core_classes(ROOT)
class_by_filename = {str(row.get('filename', '')): canonical_qid(row.get('wikidata_id', '')) for row in core_class_rows}

base_scope_map = {
    'person': class_by_filename.get('persons', ''),
    'organization': class_by_filename.get('organizations', ''),
    'episode': class_by_filename.get('episodes', ''),
    'season': class_by_filename.get('series', ''),
    'topic': class_by_filename.get('topics', ''),
    'broadcasting_program': class_by_filename.get('broadcasting_programs', ''),
}

fallback_mention_type_snapshot = config.get('_fallback_enabled_mention_types_snapshot')
assert_mention_type_snapshot_unchanged(
    config.get('fallback_enabled_mention_types', {}),
    fallback_mention_type_snapshot,
    context='building class-scope hints',
)

resolved_fallback_types = config.get('fallback_enabled_mention_types_resolved')
if not isinstance(resolved_fallback_types, list):
    raise ValueError(
        "config['fallback_enabled_mention_types_resolved'] missing. Re-run the workflow configuration cell (Step 2)."
    )

enabled_mention_types = set(resolved_fallback_types)

class_scope_hints = {
    mention_type: [qid]
    for mention_type, qid in base_scope_map.items()
    if mention_type in enabled_mention_types and qid
}

print(f'Unresolved targets handed to fallback: {len(unresolved_targets)}')
print(f'Fallback enabled mention types: {sorted(enabled_mention_types)}')
print('Class-scope hints:')
for key in sorted(class_scope_hints):
    print(f'  {key}: {class_scope_hints[key]}')

## 8) Run Fallback String Matching Stage
Fallback runs only on unresolved targets and uses class-scope narrowing.

Operator switches exposed in Step 2 config:
- `fallback_prefer_class_scoped_search`: run class-scoped SPARQL label search before generic `wbsearchentities`.
- `fallback_allow_generic_search_after_class_scoped`: permit generic fallback when class-scoped search returns no hits.
- `fallback_class_scoped_search_limit`: cap per-call class-scoped result volume.

In [ ]:
from time import perf_counter

from process.candidate_generation.wikidata.fallback_matcher import (
    enqueue_eligible_fallback_qids,
    merge_stage_candidates,
    run_fallback_string_matching_stage,
)
from process.candidate_generation.wikidata.notebook_orchestrator import (
    build_fallback_budget_plan,
    resolve_heartbeat_settings,
    resolve_stage_a_queries_this_run,
)

seed_qids = {canonical_qid(seed.get('wikidata_id', '')) for seed in seeds if canonical_qid(seed.get('wikidata_id', ''))}
stage_a_queries = resolve_stage_a_queries_this_run(result.checkpoint_stats)
raw_total_query_budget = int(config.get('query_budget_remaining', config.get('max_queries_per_run', 0)) or 0)
fallback_budget_plan = build_fallback_budget_plan(raw_total_query_budget, stage_a_queries)
fallback_query_budget = int(fallback_budget_plan['remaining_budget'])
existing_heartbeat_settings = globals().get("heartbeat_settings")
heartbeat_settings = resolve_heartbeat_settings(config, existing_heartbeat_settings if isinstance(existing_heartbeat_settings, dict) else None)

fallback_mention_type_snapshot = config.get('_fallback_enabled_mention_types_snapshot')
assert_mention_type_snapshot_unchanged(
    config.get('fallback_enabled_mention_types', {}),
    fallback_mention_type_snapshot,
    context='fallback matching',
)

fallback_enabled_types = config.get('fallback_enabled_mention_types_resolved')
if not isinstance(fallback_enabled_types, list):
    raise ValueError(
        "config['fallback_enabled_mention_types_resolved'] missing. Re-run the workflow configuration cell (Step 2)."
    )

fallback_prefer_class_scoped_search = bool(config.get('fallback_prefer_class_scoped_search', False))
fallback_allow_generic_search_after_class_scoped = bool(
    config.get('fallback_allow_generic_search_after_class_scoped', True)
 )
fallback_class_scoped_search_limit = int(
    config.get('fallback_class_scoped_search_limit', config.get('fallback_search_limit', 10)) or 10
)

fallback_config = {
    'network_budget_remaining': fallback_query_budget,
    'max_queries_per_run': fallback_query_budget,
    'query_delay_seconds': float(config.get('query_delay_seconds', 1.0) or 0.0),
    'query_timeout_seconds': int(config.get('query_timeout_seconds', 30) or 30),
    'cache_max_age_days': int(config.get('cache_max_age_days', 365) or 365),
    'network_progress_every': int(config.get('network_progress_every', 50) or 0),
    'fallback_search_limit': 10,
    'fallback_search_languages': ['de', 'en'],
    'fallback_enabled_mention_types': list(fallback_enabled_types),
    'fallback_prefer_class_scoped_search': fallback_prefer_class_scoped_search,
    'fallback_allow_generic_search_after_class_scoped': fallback_allow_generic_search_after_class_scoped,
    'fallback_class_scoped_search_limit': fallback_class_scoped_search_limit,
}
print('[notebook] Step 8 start: fallback string matching')
print(f'Stage A queries used (this run): {stage_a_queries}')
fallback_budget_label = str(fallback_budget_plan['label'])
print(f'Fallback query budget remaining: {fallback_budget_label}')
print(f"Fallback progress interval: {fallback_config['network_progress_every']} calls")
print(
    '[notebook] Fallback class-scoped settings: '
    f"prefer={fallback_prefer_class_scoped_search} "
    f"allow_generic_after_scoped={fallback_allow_generic_search_after_class_scoped} "
    f"scoped_limit={fallback_class_scoped_search_limit}"
)

step8_t0 = perf_counter()
print('[notebook] -> run_fallback_string_matching_stage')
fallback_result = run_with_progress_heartbeat(
    ROOT,
    phase="step_8_fallback_string_matching",
    work_label="run_fallback_string_matching_stage",
    heartbeat_interval_seconds=int(heartbeat_settings["interval_seconds"]),
    heartbeat_window_size=int(heartbeat_settings["window_size"]),
    work_fn=lambda: run_fallback_string_matching_stage(
        ROOT,
        unresolved_targets=unresolved_targets,
        seeds=seed_qids,
        core_class_qids={qid for values in class_scope_hints.values() for qid in values},
        class_scope_hints=class_scope_hints,
        config=fallback_config,
    ),
)
step8_elapsed = perf_counter() - step8_t0
merged_candidates = merge_stage_candidates(result.discovered_candidates, fallback_result.fallback_candidates)
print(f'[notebook] Step 8 complete in {step8_elapsed:.2f}s')
print(f'Fallback candidates: {len(fallback_result.fallback_candidates)}')
print(f'Eligible for re-entry expansion: {len(fallback_result.eligible_for_expansion_qids)}')
print(f'Ineligible fallback qids: {len(fallback_result.ineligible_qids)}')
print(f'Merged stage candidates (graph authoritative): {len(merged_candidates)}')

In [ ]:
from process.candidate_generation.wikidata.notebook_orchestrator import resolve_heartbeat_settings

existing_heartbeat_settings = globals().get("heartbeat_settings")
heartbeat_settings = resolve_heartbeat_settings(config, existing_heartbeat_settings if isinstance(existing_heartbeat_settings, dict) else None)

fallback_heartbeat = emit_event_derived_heartbeat(
    ROOT,
    phase="step_8_fallback_string_matching",
    window_size=int(heartbeat_settings["window_size"]),
)
print(f"[notebook] fallback heartbeat summary: {fallback_heartbeat}")


## 9) Re-check Eligibility and Expand Eligible Fallback Discoveries
Eligible fallback QIDs are re-entered into graph expansion with the same policy checks.

In [ ]:
from process.candidate_generation.wikidata.notebook_orchestrator import resolve_heartbeat_settings

if "fallback_result" not in globals():
    raise RuntimeError(
        "Step 8 must complete before Step 9. Rerun the fallback string matching stage after any interrupt or failure."
    )

existing_heartbeat_settings = globals().get("heartbeat_settings")
heartbeat_settings = resolve_heartbeat_settings(config, existing_heartbeat_settings if isinstance(existing_heartbeat_settings, dict) else None)

reentry_summary = enqueue_eligible_fallback_qids(
    ROOT,
    candidate_qids=fallback_result.eligible_for_expansion_qids,
    seeds={canonical_qid(seed.get('wikidata_id', '')) for seed in seeds if canonical_qid(seed.get('wikidata_id', ''))},
    core_class_qids={qid for values in class_scope_hints.values() for qid in values},
    expansion_config=expansion_config,
 )
print('Fallback re-entry summary:')
print(reentry_summary)

reentry_heartbeat = emit_event_derived_heartbeat(
    ROOT,
    phase="step_9_fallback_reentry",
    window_size=int(heartbeat_settings["window_size"]),
)
print(f"[notebook] fallback re-entry heartbeat summary: {reentry_heartbeat}")

## 10) Review Deterministic Graph Artifacts
Inspect instances/triples/query inventory outputs from checkpoint materialization.

In [ ]:
import pandas as pd
from pathlib import Path

# Load deterministic materialization artifacts from projections folder
projections_dir = ROOT / "data" / "20_candidate_generation" / "wikidata" / "projections"
instances_path = projections_dir / "instances.csv"
triples_path = projections_dir / "triples.csv"
inventory_path = projections_dir / "query_inventory.csv"
class_resolution_map_path = projections_dir / "class_resolution_map.csv"

if instances_path.exists():
    instances_df = pd.read_csv(instances_path)
    print(f"Loaded {len(instances_df)} rows from {instances_path.name}\n")

    print("Instances Preview:")
    display(instances_df)

    if triples_path.exists():
        triples_df = pd.read_csv(triples_path)
        print(f"Loaded {len(triples_df)} rows from {triples_path.name}")
        display(triples_df)

    if inventory_path.exists():
        inventory_df = pd.read_csv(inventory_path)
        print(f"Loaded {len(inventory_df)} rows from {inventory_path.name}")
        display(inventory_df)

    if class_resolution_map_path.exists():
        class_resolution_map_df = pd.read_csv(class_resolution_map_path)
        print(f"Loaded {len(class_resolution_map_df)} rows from {class_resolution_map_path.name}")
        print("Subclass exploration summary:")
        if "max_depth" in class_resolution_map_df.columns and not class_resolution_map_df.empty:
            print(f"  max_depth (from artifact): {class_resolution_map_df['max_depth'].dropna().astype(int).max()}")
        if "conflict_flag" in class_resolution_map_df.columns and not class_resolution_map_df.empty:
            print(f"  conflict rows: {int(class_resolution_map_df['conflict_flag'].sum())}")
        display(class_resolution_map_df.head(50))
else:
    print(f"Materialized instances file not found at {instances_path}")


## 11) Emit Handler Materialization Benchmark Evidence (GRW-004)
Run reproducible incremental vs full-rebuild handler benchmark artifacts as notebook closeout evidence.
This benchmark is local projection/orchestration focused and does not issue additional Wikidata network calls.

In [ ]:
from process.candidate_generation.wikidata.handler_benchmark import run_handler_materialization_benchmark
from process.candidate_generation.wikidata.notebook_orchestrator import (
    build_benchmark_run_context,
    build_benchmark_settings,
    resolve_heartbeat_settings,
)

benchmark_settings = build_benchmark_settings(config)
heartbeat_settings = resolve_heartbeat_settings(config, heartbeat_settings if "heartbeat_settings" in globals() else None)

if not bool(benchmark_settings["run_enabled"]):
    print("[notebook] Step 11 skipped (set config['handler_benchmark_run']=True to enable)")
else:
    print("[notebook] Step 11 start: handler materialization benchmark evidence")
    benchmark_run_context = build_benchmark_run_context(
        config,
        target_rows_count=int(len(all_target_rows)) if "all_target_rows" in globals() else 0,
        seed_count=int(len(seeds)) if "seeds" in globals() else 0,
        unresolved_targets_count=int(len(unresolved_targets)) if "unresolved_targets" in globals() else 0,
    )
    benchmark_summary = run_with_progress_heartbeat(
        ROOT,
        phase="step_11_handler_benchmark",
        work_label="run_handler_materialization_benchmark",
        heartbeat_interval_seconds=int(heartbeat_settings["interval_seconds"]),
        heartbeat_window_size=int(heartbeat_settings["window_size"]),
        work_fn=lambda: run_handler_materialization_benchmark(
            ROOT,
            rounds=int(benchmark_settings["rounds"]),
            batch_size=int(benchmark_settings["batch_size"]),
            include_full_rebuild=bool(benchmark_settings["include_full_rebuild"]),
            run_context=benchmark_run_context,
        ),
    )

    print("[notebook] Step 11 complete")
    print(f"[notebook] Step 11 benchmark context: {benchmark_summary.get('run_context', {})}")
    print("Benchmark artifacts:")
    for key, value in benchmark_summary.get("artifacts", {}).items():
        print(f"  - {key}: {Path(value).relative_to(ROOT)}")

    print("Benchmark aggregate rows:")
    display(pd.DataFrame(benchmark_summary.get("aggregate_rows", [])))

    benchmark_heartbeat = emit_event_derived_heartbeat(
        ROOT,
        phase="step_11_handler_benchmark",
        window_size=int(heartbeat_settings["window_size"]),
    )
    print(f"[notebook] benchmark heartbeat summary: {benchmark_heartbeat}")

## 12) Emit Runtime Evidence Bundle (GRW-005 / GRW-006 / GRW-009)
Persist a structured closeout artifact that captures runtime mode/configuration and stage outcomes for reproducible validation evidence.

In [ ]:
from process.candidate_generation.wikidata.notebook_orchestrator import (
    build_runtime_evidence_inputs,
    build_runtime_evidence_payload,
 )
from process.candidate_generation.wikidata.runtime_evidence import write_notebook21_runtime_evidence

runtime_evidence_enabled = bool(config.get("runtime_evidence_bundle_enabled", True))

if not runtime_evidence_enabled:
    print("[notebook] Step 12 skipped (set config['runtime_evidence_bundle_enabled']=True to enable)")
else:
    runtime_inputs = build_runtime_evidence_inputs(
        checkpoint_stats=result.checkpoint_stats if "result" in globals() else {},
        fallback_result=fallback_result if "fallback_result" in globals() else None,
        node_integrity_result=node_integrity_result if "node_integrity_result" in globals() else None,
        reentry_summary=reentry_summary if "reentry_summary" in globals() else {},
        benchmark_summary=benchmark_summary if "benchmark_summary" in globals() else {},
    )
    benchmark_summary_payload = runtime_inputs["benchmark_summary_payload"]

    runtime_payload_parts = build_runtime_evidence_payload(
        config,
        resume_mode=str(resume_decision.get("mode", "")) if "resume_decision" in globals() else "",
        stage_a_queries_this_run=int(runtime_inputs["stage_a_queries_this_run"]),
        node_integrity_timeout_warnings=int(runtime_inputs["node_integrity_timeout_warnings"]),
        fallback_candidates_count=int(runtime_inputs["fallback_candidates_count"]),
        reentry_expanded_qids_count=int(runtime_inputs["reentry_expanded_qids_count"]),
        fallback_class_scoped_search_queries=int(runtime_inputs["fallback_class_scoped_search_queries"]),
        fallback_generic_search_queries=int(runtime_inputs["fallback_generic_search_queries"]),
        fallback_class_scoped_hits=int(runtime_inputs["fallback_class_scoped_hits"]),
        fallback_generic_hits=int(runtime_inputs["fallback_generic_hits"]),
        benchmark_summary_present=bool(benchmark_summary_payload),
    )

    runtime_evidence = write_notebook21_runtime_evidence(
        ROOT,
        run_context=runtime_payload_parts["run_context"],
        stage_summaries=runtime_payload_parts["stage_summaries"],
        phase_outcomes=runtime_payload_parts["phase_outcomes"],
        benchmark_summary=benchmark_summary_payload,
    )

    print("[notebook] Step 12 complete")
    print(
        "[notebook] Step 12 fallback mode counters: "
        f"class_scoped_queries={int(runtime_inputs['fallback_class_scoped_search_queries'])} "
        f"generic_queries={int(runtime_inputs['fallback_generic_search_queries'])} "
        f"class_scoped_hits={int(runtime_inputs['fallback_class_scoped_hits'])} "
        f"generic_hits={int(runtime_inputs['fallback_generic_hits'])}"
    )
    print("Runtime evidence artifacts:")
    for key, value in runtime_evidence.get("artifacts", {}).items():
        print(f"  - {key}: {Path(value).relative_to(ROOT)}")